In [1]:
import tensorflow as tf

tf.compat.v1.disable_v2_behavior() # 여기까지가 템플런임


Instructions for updating:
non-resource variables are not supported in the long term


In [2]:
# ANN: 입력과 출력 레이어가 하나씩인 매우 단순한 형태 => 하나의 '식'만으로 표현
# DNN: Deep한 신경망 => n개의 hidden layer를 도입, '식'의 개수가 늘어나 정확도 상승

xData = [[80, 20], [95, 5], [10, 90], [90, 10], [5, 95]]
yData = [[1,0], [1,0], [0,1], [1,0], [0,1]]
label = [1, 0]

In [ ]:
import tensorflow as tf

# 가중치 초기화
# 초기값 0 사용 방지
def weight_var(shape):
    return tf.Variable(tf.random.truncated_normal(shape, stddev=0.1, dtype=tf.float64))

def bias_var(shape):
    return tf.Variable(tf.constant(0.1, shape=shape, dtype=tf.float64))

x = tf.compat.v1.placeholder(tf.float64, shape=[None, 2])
y = tf.compat.v1.placeholder(tf.float64, shape=[None, 2])

# Layer 1: 활성함수 미적용 상태로 f1 생성 (필요 시 추가 가능)
W1 = weight_var([2, 100])
b1 = bias_var([100])
f1 = tf.nn.relu(tf.add(tf.matmul(x, W1), b1)) # 비선형성 추가

# Layer 2
W2 = weight_var([100, 50])
b2 = bias_var([50])
f2 = tf.nn.relu(tf.add(tf.matmul(f1, W2), b2))

# Layer 3: b3 차원 수정 (50 -> 80)
W3 = weight_var([50, 80])
b3 = bias_var([80])
f3 = tf.nn.relu(tf.add(tf.matmul(f2, W3), b3))

# Layer 4: 출력층 (ReLU 제거)
W4 = weight_var([80, 2])
b4 = bias_var([2])
f4 = tf.add(tf.matmul(f3, W4), b4) # Logits 상태 유지

# Loss 및 Optimizer
l = tf.reduce_mean(tf.compat.v1.nn.softmax_cross_entropy_with_logits_v2(labels=y, logits=f4))
# Softmax: '결과의 총합' = 1로 바꿔 주는 (=비율화 / 확률분포) 함수.
o = tf.compat.v1.train.AdamOptimizer(learning_rate=0.01)
g = o.minimize(l)

s = tf.compat.v1.Session()
s.run(tf.compat.v1.global_variables_initializer())

# 학습 루프
for epoch in range(1000):
    s.run(g, feed_dict={x: xData, y: yData})
    if epoch % 100 == 0:
        loss_val = s.run(l, feed_dict={x: xData, y: yData})
        print(f"Epoch: {epoch}, Loss: {loss_val}")

Epoch: 0, Loss: 0.023181876748574644
Epoch: 100, Loss: 0.0
Epoch: 200, Loss: 0.0
Epoch: 300, Loss: 0.0
Epoch: 400, Loss: 0.0
Epoch: 500, Loss: 0.0
Epoch: 600, Loss: 0.0
Epoch: 700, Loss: 0.0
Epoch: 800, Loss: 0.0
Epoch: 900, Loss: 0.0


In [8]:
import numpy as np

# 1. 예측값 계산 (Softmax 적용)
# f4는 Logit이므로 확률로 변환하기 위해 Softmax를 적용합니다.
predict_op = tf.nn.softmax(f4)

# 2. 정확도 계산 로직
# argmax를 통해 가장 높은 확률을 가진 인덱스를 추출하여 실제 레이블과 비교합니다.
is_correct = tf.equal(tf.argmax(predict_op, 1), tf.argmax(y, 1))
accuracy_op = tf.reduce_mean(tf.cast(is_correct, tf.float64))

# 3. 테스트 실행
print("\n--- Test Results ---")
final_predict, final_acc = s.run([predict_op, accuracy_op], feed_dict={x: xData, y: yData})

for i in range(len(xData)):
    predicted_class = np.argmax(final_predict[i])
    actual_class = np.argmax(yData[i])
    print(f"Input: {xData[i]} | Actual: {actual_class} | Predicted: {predicted_class} | Prob: {final_predict[i]}")

print(f"\nFinal Accuracy: {final_acc * 100}%")


--- Test Results ---
Input: [80, 20] | Actual: 0 | Predicted: 0 | Prob: [1.00000000e+00 9.95323309e-33]
Input: [95, 5] | Actual: 0 | Predicted: 0 | Prob: [1.00000000e+00 3.38427145e-41]
Input: [10, 90] | Actual: 1 | Predicted: 1 | Prob: [6.16074486e-34 1.00000000e+00]
Input: [90, 10] | Actual: 0 | Predicted: 0 | Prob: [1.00000000e+00 1.62273777e-38]
Input: [5, 95] | Actual: 1 | Predicted: 1 | Prob: [6.23561032e-38 1.00000000e+00]

Final Accuracy: 100.0%
